# Amalgam 2026 — Polymer Property Prediction (fast configuration)

Self-contained: loading, featurization, cross-validation, training, inference and
`submission.csv` all run in this single notebook pass. No external data, no
pretrained weights, no uploaded artifacts. Seeds fixed throughout.

**Metric:** `Score = (R2_Tg + R2_Egc) / 2` — both targets weighted equally.

**Features.** The rows are polymer *repeat units* carrying exactly two `*`
attachment points, not ordinary molecules. Alongside standard RDKit descriptors,
Morgan counts, MACCS keys and Avalon counts, we build 18 **polymer-aware**
features from the backbone path between the two attachment points: its length,
rotatable bonds, ring/aromatic content, conjugation runs, and side-chain count
and mass fraction. Backbone length alone correlates +0.47 with Tg. Those 18
columns contribute as much as 1024 Avalon bits.

**Models.** `tg` and `egc` are disjoint row sets, so each gets its own models.
LightGBM / XGBoost / CatBoost train on the full feature set; Ridge trains on
dense descriptors only (sparse fingerprints destabilise it badly). Blend weights
are fitted per target on out-of-fold predictions.

In [ ]:
import os, random, warnings, glob, importlib, subprocess, sys
import numpy as np
import pandas as pd

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore")

N_FOLDS = 10   # +0.0019 on held-out data (src/foldstudy.py); doubles runtime
TARGET_TYPES = ["tg", "egc"]
GNN_EPOCHS, GNN_FOLDS = 120, 10
# Three seeds averaged. GBDT seed averaging measured +0.0000, but neural
# nets have real initialisation variance: GNN solo 0.9066 -> 0.9151, and
# the blend 0.9211 -> 0.9231. Diminishing fast though - 6 seeds added only
# +0.0006 more and pushed the notebook past Kaggle's 9h limit.
# 10 folds beats 5: each model trains on 90% of rows instead of 80%, worth
# +0.0039 solo / +0.0017 in blend. Two seeds rather than three keeps the
# runtime inside Kaggle's 9h limit at the same total training cost.
GNN_SEEDS = [SEED, SEED + 100]
MORGAN_BITS, MORGAN_RADIUS, AVALON_SIZE = 2048, 2, 1024

In [ ]:
# Dependency bootstrap, checked up front so a missing package fails in seconds
# rather than partway through an expensive run. Open-source libraries only.
REQUIRED = ["rdkit", "lightgbm", "xgboost", "catboost", "sklearn", "scipy",
            "joblib", "torch"]
PIP_NAME = {"sklearn": "scikit-learn"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()
for mod in REQUIRED:
    importlib.import_module(mod)

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, MACCSkeys, rdMolDescriptors, rdmolops
from rdkit.Chem import rdFingerprintGenerator as rfg
from rdkit.Avalon import pyAvalonTools
RDLogger.DisableLog("rdApp.*")

import rdkit
print("dependencies OK | rdkit", rdkit.__version__)

## 1. Data

In [ ]:
def find_data_dir():
    hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if not hits:
        raise FileNotFoundError("train.csv not found; have: "
                                + str(glob.glob("/kaggle/input/*")))
    d = os.path.dirname(sorted(hits, key=len)[0])
    print("data dir:", d)
    return d

IN = find_data_dir()
train = pd.read_csv(f"{IN}/train.csv")
test = pd.read_csv(f"{IN}/test.csv")
sample_sub = pd.read_csv(f"{IN}/sample_submission.csv")
print("train", train.shape, "| test", test.shape)
print(train.groupby("target_type").target.agg(["count","mean","std","min","max"]).round(3))

## 2. Polymer-aware features

The backbone is the shortest path between the two `*` attachment points;
everything hanging off it is side chain. Chain stiffness drives Tg and
conjugation along the chain drives Egc, so both are measured explicitly.

In [ ]:
def attachment_points(mol):
    return [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() == 0]


def backbone_path(mol):
    d = attachment_points(mol)
    if len(d) != 2:
        return None
    try:
        return list(rdmolops.GetShortestPath(mol, d[0], d[1]))
    except Exception:
        return None


def _longest_true_run(flags):
    best = cur = 0
    for f in flags:
        cur = cur + 1 if f else 0
        best = max(best, cur)
    return best


POLY_NAMES = ["bb_len","bb_frac","side_atoms","side_frac","bb_rot","bb_ring_atoms",
              "bb_aromatic","bb_hetero","bb_conj_bonds","bb_conj_frac","bb_max_conj_run",
              "bb_sp3_frac","n_side_chains","max_side_len","side_mass_frac",
              "bb_double_bonds","flexibility","bb_branch_pts"]


def backbone_scalars(mol):
    path = backbone_path(mol)
    if path is None:
        return [np.nan] * len(POLY_NAMES)

    n_atoms = mol.GetNumAtoms()
    bb = set(path)
    bb_atoms = [mol.GetAtomWithIdx(i) for i in path]
    bonds = [mol.GetBondBetweenAtoms(path[i], path[i+1]) for i in range(len(path)-1)]
    bonds = [b for b in bonds if b is not None]
    conj = [b.GetIsConjugated() for b in bonds]

    rot = sum(1 for b in bonds
              if b.GetBondType() == Chem.BondType.SINGLE and not b.IsInRing()
              and b.GetBeginAtom().GetDegree() > 1 and b.GetEndAtom().GetDegree() > 1)

    side_sizes, seen = [], set()
    for a in bb_atoms:
        for nb in a.GetNeighbors():
            if nb.GetIdx() in bb or nb.GetIdx() in seen:
                continue
            stack, comp = [nb.GetIdx()], set()
            while stack:
                cur = stack.pop()
                if cur in comp or cur in bb:
                    continue
                comp.add(cur)
                stack += [x.GetIdx() for x in mol.GetAtomWithIdx(cur).GetNeighbors()]
            seen |= comp
            if comp:
                side_sizes.append(len(comp))

    side_atoms = n_atoms - len(path)
    bb_mass = sum(a.GetMass() for a in bb_atoms)
    tot_mass = sum(a.GetMass() for a in mol.GetAtoms()) or 1.0

    return [
        len(path),
        len(path)/n_atoms if n_atoms else np.nan,
        side_atoms,
        side_atoms/n_atoms if n_atoms else np.nan,
        rot,
        sum(1 for a in bb_atoms if a.IsInRing()),
        sum(1 for a in bb_atoms if a.GetIsAromatic()),
        sum(1 for a in bb_atoms if a.GetAtomicNum() not in (6, 0)),
        sum(conj),
        (sum(conj)/len(conj)) if conj else np.nan,
        _longest_true_run(conj),
        (sum(1 for a in bb_atoms if a.GetHybridization() == Chem.HybridizationType.SP3)
         / len(path)) if path else np.nan,
        len(side_sizes),
        max(side_sizes) if side_sizes else 0,
        1.0 - bb_mass/tot_mass,
        sum(1 for b in bonds if b.GetBondType() == Chem.BondType.DOUBLE),
        rdMolDescriptors.CalcNumRotatableBonds(mol)/max(n_atoms, 1),
        sum(1 for a in bb_atoms if a.GetDegree() > 2),
    ]


def backbone_atoms_ring_complete(mol):
    """Backbone path expanded so any ring it passes through is included whole.
    Chemically right (a phenylene in the chain is a backbone unit) and it keeps
    the extracted fragment sanitisable: path-only extraction breaks aromatic
    rings and fails on ~70% of these molecules."""
    path = backbone_path(mol)
    if path is None:
        return None
    bb = set(path)
    rings = mol.GetRingInfo().AtomRings()
    changed = True
    while changed:
        changed = False
        for ring in rings:
            r = set(ring)
            if bb & r and not r <= bb:
                bb |= r
                changed = True
    return bb


def _submol(mol, keep):
    rw = Chem.RWMol(mol)
    for i in sorted((a.GetIdx() for a in mol.GetAtoms() if a.GetIdx() not in keep),
                    reverse=True):
        rw.RemoveAtom(i)
    m = rw.GetMol()
    try:
        Chem.SanitizeMol(m)
        return m
    except Exception:
        return None


_DESC_REF = None

def backbone_descriptors(mol):
    """Descriptors of the backbone ALONE. The 18 scalars summarise the chain;
    these describe it as a molecule in its own right (+0.0025 at 3 folds, mostly
    on tg, which fits: glass transition is a chain property)."""
    global _DESC_REF
    if _DESC_REF is None:
        _DESC_REF = list(Descriptors.CalcMolDescriptors(Chem.MolFromSmiles("C")).keys())
    keep = backbone_atoms_ring_complete(mol)
    bb = _submol(mol, keep) if keep is not None else None
    if bb is None:
        return [np.nan] * len(_DESC_REF)
    try:
        return list(Descriptors.CalcMolDescriptors(bb).values())
    except Exception:
        return [np.nan] * len(_DESC_REF)


## 3. Full featurizer

Dummy atoms break Gasteiger-charge and BCUT descriptors, so descriptors are
computed twice: on the raw molecule and on a `*`→carbon capped copy. The polymer
block uses the raw molecule, since capping would destroy the attachment points
it depends on.

In [ ]:
_MGEN = None

def morgan_gen():
    global _MGEN
    if _MGEN is None:
        _MGEN = rfg.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_BITS)
    return _MGEN


def cap_wildcards(mol):
    rw = Chem.RWMol(mol)
    for atom in rw.GetAtoms():
        if atom.GetAtomicNum() == 0:
            atom.SetAtomicNum(6); atom.SetNoImplicit(False); atom.SetNumExplicitHs(0)
    m = rw.GetMol()
    try:
        Chem.SanitizeMol(m)
    except Exception:
        return None
    return m


def featurize_one(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    capped = cap_wildcards(mol)
    fp_mol = capped if capped is not None else mol
    vals, names = [], []

    d_raw = Descriptors.CalcMolDescriptors(mol)
    vals += list(d_raw.values()); names += [f"draw_{k}" for k in d_raw]

    if capped is not None:
        d_cap = Descriptors.CalcMolDescriptors(capped)
        vals += list(d_cap.values()); names += [f"dcap_{k}" for k in d_cap]
    else:
        vals += [np.nan]*len(d_raw); names += [f"dcap_{k}" for k in d_raw]

    mg = morgan_gen().GetCountFingerprintAsNumPy(fp_mol).astype(np.float32)
    vals += mg.tolist(); names += [f"mg2_{i}" for i in range(len(mg))]

    maccs = np.array(MACCSkeys.GenMACCSKeys(fp_mol), dtype=np.float32)
    vals += maccs.tolist(); names += [f"maccs_{i}" for i in range(len(maccs))]

    avl = np.array(pyAvalonTools.GetAvalonCountFP(fp_mol, nBits=AVALON_SIZE).ToList(),
                   dtype=np.float32)
    vals += avl.tolist(); names += [f"avl_{i}" for i in range(len(avl))]

    vals += backbone_scalars(mol); names += [f"poly_{n}" for n in POLY_NAMES]

    bbd = backbone_descriptors(mol)
    vals += bbd
    if _DESC_REF is not None:
        names += [f"bbd_{k}" for k in _DESC_REF]
    return vals, names

In [ ]:
%%time
from joblib import Parallel, delayed

all_smiles = pd.Index(pd.concat([train.smiles, test.smiles]).unique(), name="smiles")
print(f"featurizing {len(all_smiles)} unique SMILES ...")
out = Parallel(n_jobs=-1, verbose=1)(delayed(featurize_one)(s) for s in all_smiles)

n_failed = sum(o is None for o in out)
names = next(o[1] for o in out if o is not None)
rows = [o[0] if o is not None else [np.nan]*len(names) for o in out]

FEAT = pd.DataFrame(rows, columns=names, index=all_smiles, dtype=np.float32)
FEAT = FEAT.replace([np.inf, -np.inf], np.nan)
FEAT = FEAT[FEAT.columns[FEAT.nunique(dropna=False) > 1]]
print(f"features: {FEAT.shape} | parse failures: {n_failed}")

TREE_PREFIXES  = ("draw_", "dcap_", "mg2_", "maccs_", "avl_", "poly_", "bbd_")
RIDGE_PREFIXES = ("draw_", "dcap_")          # linear model: dense descriptors only
TREE_COLS  = [c for c in FEAT.columns if c.startswith(TREE_PREFIXES)]
RIDGE_COLS = [c for c in FEAT.columns if c.startswith(RIDGE_PREFIXES)]
print(f"tree features: {len(TREE_COLS)} | ridge features: {len(RIDGE_COLS)}")

## 4. Metric and cross-validation

In [ ]:
from sklearn.model_selection import KFold

def r2(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    return 1.0 - ((y_true-y_pred)**2).sum() / ((y_true-y_true.mean())**2).sum()


def run_cv(make_model, X, y, X_test, n_folds=N_FOLDS, seed=SEED):
    X = np.asarray(X, np.float32); y = np.asarray(y, float)
    X_test = np.asarray(X_test, np.float32)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros(len(y)); test_preds = np.zeros((n_folds, len(X_test)))
    for i, (tr_i, va_i) in enumerate(kf.split(X)):
        m = make_model()
        m.fit_fold(X[tr_i], y[tr_i], X[va_i], y[va_i])
        oof[va_i] = m.predict(X[va_i]); test_preds[i] = m.predict(X_test)
        print(f"    fold {i}: r2={r2(y[va_i], oof[va_i]):.4f}")
    print(f"    OOF r2 = {r2(y, oof):.4f}")
    return oof, test_preds.mean(axis=0)

## 5. Models

In [ ]:
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostRegressor

class LGBMWrap:
    def __init__(self, seed=SEED):
        self.m = lgb.LGBMRegressor(n_estimators=4000, learning_rate=0.03, num_leaves=64,
            min_child_samples=20, colsample_bytree=0.6, subsample=0.8, subsample_freq=1,
            reg_lambda=1.0, verbose=-1, n_jobs=-1, random_state=seed)
    def fit_fold(self, Xtr, ytr, Xva, yva):
        self.m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="rmse",
                   callbacks=[lgb.early_stopping(200, verbose=False)])
    def predict(self, X): return self.m.predict(X)

class XGBWrap:
    def __init__(self, seed=SEED):
        self.m = xgb.XGBRegressor(n_estimators=4000, learning_rate=0.03, max_depth=7,
            min_child_weight=5, colsample_bytree=0.5, subsample=0.8, reg_lambda=1.0,
            tree_method="hist", n_jobs=-1, early_stopping_rounds=200, random_state=seed)
    def fit_fold(self, Xtr, ytr, Xva, yva):
        self.m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    def predict(self, X): return self.m.predict(X)

class CatWrap:
    def __init__(self, seed=SEED):
        self.m = CatBoostRegressor(iterations=4000, learning_rate=0.03, depth=8,
            l2_leaf_reg=3.0, loss_function="RMSE", verbose=0,
            allow_writing_files=False, random_seed=seed)
    def fit_fold(self, Xtr, ytr, Xva, yva):
        self.m.fit(np.nan_to_num(Xtr), ytr, eval_set=(np.nan_to_num(Xva), yva),
                   early_stopping_rounds=200)
    def predict(self, X): return self.m.predict(np.nan_to_num(X))

class RidgeWrap:
    """Weak alone (~0.82) but decorrelated from the trees, so it earns blend
    weight. Trained on dense descriptors only: sparse fingerprints send it to
    R2 = -1.39 through rare-feature amplification after standardisation."""
    def __init__(self, seed=SEED, alpha=10.0):
        from sklearn.linear_model import Ridge
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        from sklearn.impute import SimpleImputer
        self.m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                               Ridge(alpha=alpha, random_state=seed))
    def fit_fold(self, Xtr, ytr, Xva, yva): self.m.fit(Xtr, ytr)
    def predict(self, X): return self.m.predict(X)

MODELS = {"lgbm": (LGBMWrap, "tree"), "xgb": (XGBWrap, "tree"),
          "cat": (CatWrap, "tree"), "ridge": (RidgeWrap, "ridge")}

## 6. Train per target, blend, submit

## 6. Graph neural network

The one model that reads raw molecular topology rather than topology hashed into
fingerprint bits. Trained from scratch (pretrained weights are banned by rule
6.2.1). Its errors correlate only 0.73-0.79 with the boosted trees, so it earns
large blend weight: 0.29 on tg and 0.44 on egc.

Pinned to a single thread deliberately. Parallel `index_add_` sums atomics in
varying order, which made two identical runs differ by up to 0.18 per
prediction. Rule 7.2 has the hosts re-run this notebook and voids the submission
if the score does not reproduce, so exact determinism is required. These graphs
are small enough that threading bought nothing.

In [ ]:
import torch
import torch.nn as nn
from rdkit.Chem import rdmolops as _rdmolops

torch.manual_seed(SEED)
torch.set_num_threads(1)          # determinism, see note above

GNN_ATOMS = [6, 7, 8, 9, 14, 15, 16, 17, 35, 53, 0]     # 0 = '*' attachment point
GNN_HYB = [Chem.HybridizationType.SP, Chem.HybridizationType.SP2,
           Chem.HybridizationType.SP3, Chem.HybridizationType.SP3D]


def atom_features(a):
    z = a.GetAtomicNum()
    f = [1.0 if z == x else 0.0 for x in GNN_ATOMS] + [1.0 if z not in GNN_ATOMS else 0.0]
    f += [1.0 if a.GetHybridization() == h else 0.0 for h in GNN_HYB]
    f += [a.GetDegree() / 4.0, a.GetFormalCharge(), float(a.GetIsAromatic()),
          float(a.IsInRing()), a.GetTotalNumHs() / 4.0, a.GetMass() / 200.0]
    return f


N_ATOM_FEAT = len(atom_features(Chem.MolFromSmiles("C").GetAtomWithIdx(0)))


def mol_to_graph(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    x = torch.tensor([atom_features(a) for a in m.GetAtoms()], dtype=torch.float32)
    src, dst = [], []
    for b in m.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        src += [i, j]; dst += [j, i]
    if not src:
        src, dst = [0], [0]
    return x, torch.tensor([src, dst], dtype=torch.long)


class MPNN(nn.Module):
    def __init__(self, hidden=128, layers=3, dropout=0.1):
        super().__init__()
        self.inp = nn.Linear(N_ATOM_FEAT, hidden)
        self.msg = nn.ModuleList(nn.Linear(hidden, hidden) for _ in range(layers))
        self.upd = nn.ModuleList(nn.GRUCell(hidden, hidden) for _ in range(layers))
        self.norm = nn.ModuleList(nn.LayerNorm(hidden) for _ in range(layers))
        self.head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, x, edge, batch, n_graphs):
        h = torch.relu(self.inp(x))
        src, dst = edge
        for msg, upd, norm in zip(self.msg, self.upd, self.norm):
            agg = torch.zeros_like(h).index_add_(0, dst, msg(h)[src])
            h = norm(upd(agg, h))
        mean = torch.zeros(n_graphs, h.shape[1]).index_add_(0, batch, h)
        cnt = torch.zeros(n_graphs).index_add_(
            0, batch, torch.ones(len(batch))).clamp(min=1).unsqueeze(1)
        mean = mean / cnt
        mx = torch.full((n_graphs, h.shape[1]), -1e9)
        mx = mx.index_reduce_(0, batch, h, "amax", include_self=True)
        return self.head(torch.cat([mean, mx], 1)).squeeze(-1)


def gnn_collate(graphs, idx):
    xs, edges, batch, off = [], [], [], 0
    for k, i in enumerate(idx):
        x, e = graphs[i]
        xs.append(x); edges.append(e + off)
        batch.append(torch.full((len(x),), k, dtype=torch.long))
        off += len(x)
    return torch.cat(xs), torch.cat(edges, 1), torch.cat(batch), len(idx)


def run_gnn(smiles_tr, y_raw, smiles_te, epochs=120, hidden=128, bs=64,
            lr=1e-3, n_folds=5, seeds=None):
    """OOF + test predictions for one target, averaged over seeds."""
    seeds = seeds if seeds is not None else [SEED]
    g_tr = [mol_to_graph(s) for s in smiles_tr]
    g_te = [mol_to_graph(s) for s in smiles_te]
    ok = np.array([g is not None for g in g_tr])
    mu, sd = y_raw[ok].mean(), y_raw[ok].std()
    y = (y_raw - mu) / sd
    yt = torch.tensor(y, dtype=torch.float32)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    oof_seeds, test_preds = [], []

    for si, base_seed in enumerate(seeds):
      oof = np.full(len(y), np.nan)
      for fold, (tr_i, va_i) in enumerate(kf.split(np.arange(len(y)))):
        tr_i = tr_i[ok[tr_i]]; va_ok = va_i[ok[va_i]]
        torch.manual_seed(base_seed + fold)
        rng = np.random.default_rng(base_seed + fold)
        model = MPNN(hidden=hidden)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=lr, total_steps=epochs * max(1, len(tr_i) // bs))
        best, best_state, patience = 1e9, None, 0
        for ep in range(epochs):
            model.train()
            perm = rng.permutation(tr_i)
            for k in range(0, len(perm), bs):
                b = perm[k:k + bs]
                if len(b) < 2:
                    continue
                x, e, bt, n = gnn_collate(g_tr, b)
                loss = nn.functional.mse_loss(model(x, e, bt, n), yt[b])
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
                if sched.last_epoch < sched.total_steps - 1:
                    sched.step()
            if ep % 5 == 4 or ep == epochs - 1:
                model.eval()
                with torch.no_grad():
                    x, e, bt, n = gnn_collate(g_tr, va_ok)
                    p = model(x, e, bt, n).numpy()
                mse = ((p - y[va_ok]) ** 2).mean()
                if mse < best - 1e-5:
                    best = mse
                    best_state = {k2: v.clone() for k2, v in model.state_dict().items()}
                    patience = 0
                else:
                    patience += 1
                    if patience >= 6:
                        break
        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            x, e, bt, n = gnn_collate(g_tr, va_ok)
            oof[va_ok] = model(x, e, bt, n).numpy()
            xt, et, btt, nt = gnn_collate(g_te, range(len(g_te)))
            test_preds.append(model(xt, et, btt, nt).numpy())
        print(f"    gnn seed{si} fold {fold}: r2={r2(y_raw[va_ok], oof[va_ok]*sd+mu):.4f}")
      oof_seeds.append(oof)

    oof = np.nanmean(np.vstack(oof_seeds), axis=0)
    oof = np.nan_to_num(oof, nan=0.0) * sd + mu
    return oof, np.mean(test_preds, axis=0) * sd + mu

In [ ]:
%%time
from scipy.optimize import minimize

def fit_blend_weights(oofs, y):
    A = np.column_stack(oofs)
    if A.shape[1] == 1:
        return np.ones(1)
    obj = lambda w: np.sqrt(((A @ w - y)**2).mean())
    r = minimize(obj, np.full(A.shape[1], 1/A.shape[1]), method="SLSQP",
                 bounds=[(0.0, 1.0)]*A.shape[1],
                 constraints=[{"type":"eq","fun":lambda w: w.sum()-1}])
    w = np.clip(r.x, 0, None)
    return w / w.sum()

# Ridge dropped: it earned ~0 blend weight, and at 10 folds one
# ill-conditioned fold predicted -17080 (true range -118..490), which
# would corrupt the OOF-fitted blend weights. Removing it scores the
# same and removes the failure mode.
# Fast configuration. CatBoost is 91% of the runtime and worth only
# 0.0006 honest OOF; XGBoost contributes 0.0000. Dropping both takes the
# notebook from ~5h to ~25min for -0.0006, which is the right trade under
# rule 7.2: the hosts must re-run this and a shorter run is far safer.
USE = ["lgbm"]
pieces, per_target_r2 = [], {}

for tt in TARGET_TYPES:
    tr_t = train[train.target_type == tt].reset_index(drop=True)
    te_t = test[test.target_type == tt].reset_index(drop=True)
    y = tr_t.target.values
    print(f"\n=== {tt}: {len(tr_t)} train / {len(te_t)} test ===")

    oofs, tests, used = [], [], []
    for name in USE:
        cls, kind = MODELS[name]
        cols = TREE_COLS if kind == "tree" else RIDGE_COLS
        Xtr = FEAT.reindex(tr_t.smiles.values)[cols].values
        Xte = FEAT.reindex(te_t.smiles.values)[cols].values
        print(f"  [{name}] {Xtr.shape[1]} features")
        o, t = run_cv(lambda c=cls: c(seed=SEED), Xtr, y, Xte)
        oofs.append(o); tests.append(t); used.append(name)

    # Graph network: reads molecular topology directly, so it takes SMILES
    # rather than the feature matrix. Its errors correlate only 0.73-0.79 with
    # the trees, which is why it earns large blend weight (+0.0053 honest).
    print("  [gnn] message passing on molecular graphs")
    o, t = run_gnn(tr_t.smiles.values, y, te_t.smiles.values,
                   epochs=GNN_EPOCHS, n_folds=GNN_FOLDS, seeds=GNN_SEEDS)
    oofs.append(o); tests.append(t); used.append("gnn")

    w = fit_blend_weights(oofs, y)
    blend_oof = np.column_stack(oofs) @ w
    blend_test = np.clip(np.column_stack(tests) @ w, y.min(), y.max())
    per_target_r2[tt] = r2(y, blend_oof)
    print(f"  weights {dict(zip(used, np.round(w,3)))} -> OOF r2 = {per_target_r2[tt]:.4f}")
    pieces.append(pd.DataFrame({"id": te_t.id.values, "target": blend_test}))

print("\n" + "="*52)
for tt in TARGET_TYPES:
    print(f"  R2_{tt:4s} = {per_target_r2[tt]:.4f}")
print(f"  OFFICIAL OOF SCORE = {(per_target_r2['tg']+per_target_r2['egc'])/2:.4f}")

In [ ]:
submission = pd.concat(pieces).sort_values("id").reset_index(drop=True)
assert list(submission.columns) == list(sample_sub.columns), "column mismatch"
assert len(submission) == len(test), f"{len(submission)} rows, expected {len(test)}"
assert submission.id.is_unique and set(submission.id) == set(test.id), "id mismatch"
assert np.isfinite(submission.target).all(), "non-finite predictions"
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv", submission.shape)
submission.head()